In [1]:
pip install gradio pillow numpy


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pickle
import numpy as np
import gradio as gr
from PIL import Image, ImageOps

# =========================================================
# 1) CLASS DEFINITIONS (để pickle load được)
# =========================================================

class LinearSoftmaxClassifier:
    def __init__(self, n_features, n_classes, seed=42):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0.0, 0.01, size=(n_features, n_classes)).astype(np.float32)

    @staticmethod
    def softmax(logits):
        logits = logits - np.max(logits, axis=1, keepdims=True)
        exp_logits = np.exp(logits)
        return exp_logits / (np.sum(exp_logits, axis=1, keepdims=True) + 1e-12)

    def predict_proba(self, X):
        logits = X @ self.W
        return self.softmax(logits)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


class AdaBoostSAMME:
    def __init__(self, *args, **kwargs):
        pass

    def predict(self, X):
        n = X.shape[0]
        scores = np.zeros((n, self.n_classes), dtype=np.float64)
        for clf, alpha in zip(self.weak_learners, self.alphas):
            pred = clf.predict(X)
            scores[np.arange(n), pred] += alpha
        return np.argmax(scores, axis=1)

# =========================================================
# 2) LOAD BUNDLE
# =========================================================

BUNDLE_PATH = "adaboost_bundle.pkl"

with open(BUNDLE_PATH, "rb") as f:
    bundle = pickle.load(f)

model = bundle["model"]
mu = bundle["mu"].astype(np.float32)   # (1,784)
std = bundle["std"].astype(np.float32) # (1,784)
add_bias = bool(bundle.get("add_bias", True))

# =========================================================
# 3) UTIL: gradio input -> PIL
# =========================================================

def gradio_to_pil(x):
    """
    Gradio Sketchpad/ImageEditor có thể trả:
    - PIL.Image
    - numpy array
    - dict {"composite": ...} hoặc {"image": ...}
    """
    if x is None:
        return None

    if isinstance(x, dict):
        # ưu tiên composite
        for k in ["composite", "image", "background"]:
            if k in x and x[k] is not None:
                x = x[k]
                break
        else:
            # fallback: lấy value đầu tiên không None
            for v in x.values():
                if v is not None:
                    x = v
                    break

    if isinstance(x, np.ndarray):
        if x.dtype != np.uint8:
            x = np.clip(x, 0, 255).astype(np.uint8)
        return Image.fromarray(x)

    if isinstance(x, Image.Image):
        return x

    try:
        return Image.fromarray(np.array(x))
    except Exception:
        return None

# =========================================================
# 4) PREPROCESS: MNIST-like (invert + crop/center/pad + resize)
# =========================================================

def preprocess_mnist_like(pil_img: Image.Image, invert=True, threshold=200):
    """
    Output:
      - img28 (PIL, L) nền đen chữ trắng
      - x784 float32 shape (1,784) đã /255
    """
    img = pil_img.convert("L")

    # nếu ảnh là nền trắng chữ đen -> invert để thành MNIST
    if invert:
        img = ImageOps.invert(img)

    arr = np.array(img, dtype=np.uint8)

    # mask vùng chữ (pixel sáng)
    mask = arr > threshold

    if mask.sum() >= 10:
        ys, xs = np.where(mask)
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        cropped = img.crop((x0, y0, x1, y1))

        # pad thành vuông
        w, h = cropped.size
        side = max(w, h)
        padded = Image.new("L", (side, side), color=0)  # nền đen
        padded.paste(cropped, ((side - w) // 2, (side - h) // 2))
        img28 = padded.resize((28, 28))
    else:
        # fallback nếu không bắt được mask
        img28 = img.resize((28, 28))

    x = (np.array(img28, dtype=np.float32) / 255.0).reshape(1, -1)  # (1,784)
    return img28, x

# =========================================================
# 5) PREDICT
# =========================================================

def predict(sketch, invert=False, threshold=200):
    pil = gradio_to_pil(sketch)
    if pil is None:
        return "Chưa có hình", None

    img28, x = preprocess_mnist_like(pil, invert=invert, threshold=threshold)

    # standardize đúng theo train
    x = (x - mu) / (std + 1e-8)

    # add bias đúng shape
    if add_bias:
        x = np.hstack([x, np.ones((1, 1), dtype=np.float32)])  # (1,785)

    pred = int(model.predict(x)[0])

    debug = (
        f"Pred: {pred}\n"
        f"x shape: {x.shape}\n"
        f"x stats: min={float(x.min()):.3f} max={float(x.max()):.3f} mean={float(x.mean()):.3f} std={float(x.std()):.3f}\n"
    )
    return debug, img28

# =========================================================
# 6) UI
# =========================================================

with gr.Blocks() as demo:
    gr.Markdown("## AdaBoost MNIST — Vẽ số để test (dùng adaboost_bundle.pkl)")

    with gr.Row():
        canvas = gr.Sketchpad(label="Vẽ số (chuột)", height=300, width=300)
        out_text = gr.Textbox(label="Kết quả / debug", lines=6)
        out_img = gr.Image(label="28×28 sau preprocess", image_mode="L")

    with gr.Row():
        invert_chk = gr.Checkbox(value=False, label="Invert (nền trắng chữ đen -> MNIST)")
        thr = gr.Slider(0, 255, value=200, step=1, label="Threshold crop (tăng nếu nền sáng quá)")

    btn = gr.Button("Predict")
    btn.click(fn=predict, inputs=[canvas, invert_chk, thr], outputs=[out_text, out_img])

demo.launch()


c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
